In [ ]:
   import json
   import os
   import pandas as pd

In [ ]:
   ordner = "../data/toxic_and_baseline_random_oss"
   dateien = os.listdir(ordner)
   print("Anzahl Gespräche:", len(dateien))
   print(dateien[:5])

In [ ]:
   with open(os.path.join(ordner, dateien[0]), encoding="utf-8") as f:
       gespraech = json.load(f)

   print("Thema:", gespraech["proposition"])
   print("Anzahl Runden:", gespraech["nrounds"])
   print("Einigung erreicht:", gespraech["in_alignment"])
   

In [ ]:
   for seite, agenten in gespraech["active_agents"].items():
       for a in agenten:
           print(seite.upper(), "->", a["description"])

In [ ]:
   for s in gespraech["opening_statements"]:
       print(s["agent"], ":")
       print(s["opening_statement"][:500], "...\n")

In [ ]:
   for runde, antworten in gespraech["discussion_history"].items():
       if runde == "opening_statements":
           continue
       print("=====", runde, "=====")
       for a in antworten:
           print(a["agent"], ":")
           print(a["next_answer"][:500], "...\n")

In [ ]:
   zeilen = []
   for name in dateien:
       with open(os.path.join(ordner, name), encoding="utf-8") as f:
           g = json.load(f)
       toxizitaet = name.split("toxic_level_")[1].replace(".json", "")
       zeilen.append({
           "datei": name,
           "thema": g["proposition"],
           "toxizitaet": toxizitaet,
           "runden": g["nrounds"],
           "einigung": g["in_alignment"],
       })

   df = pd.DataFrame(zeilen)
   df.head()

In [ ]:
   df.groupby("toxizitaet").agg(
       anzahl=("datei", "count"),
       runden_durchschnitt=("runden", "mean"),
       einigungsquote=("einigung", "mean"),
   ).round(2)

   ## Beobachtungen (17.10.2026)

   - **Runden:** Ohne Toxizität dauert eine Debatte im Schnitt 2,1 Runden, bei heavy 4,3 Runden → toxische Debatten dauern länger (bestätigt MAD-Paper).
   - **Einigungsquote:** Ohne Toxizität 35 % Einigung, bei heavy nur 16 % → Toxizität erschwert Einigung.
   - **Offene Frage an Prof. Mangold:** Warum enden so viele Debatten ohne Einigung? Wichtig für H2 (Gewinner nur bei Einigung messbar).
   - **Personas:** Manche Personas haben schon Namen (z. B. „Jamie Reed“) → können Geschlecht andeuten, beim Gender-Parameter beachten.
   - **Toxische Sprache:** Sarkasmus, Lächerlich machen, Herablassung → mögliche Kategorien für H3.